# K-Means + Agglomerative clustering on `combined_cleaned.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score, pairwise_distances_argmin_min
from sklearn.preprocessing import StandardScaler

from scipy.cluster.hierarchy import dendrogram, linkage

%matplotlib inline

In [ ]:
# STEP 2: Load dataset
data = pd.read_csv('combined_cleaned.csv')
print('Data shape:', data.shape)
display(data.head())

In [ ]:
# STEP 3: Build feature matrix (unsupervised -> exclude any supervised target)
if 'heart_disease' in data.columns:
    X = data.drop(columns=['heart_disease'])
else:
    X = data.copy()

# One-hot encode categoricals
X = pd.get_dummies(X, drop_first=True)
print('Encoded features shape:', X.shape)

In [ ]:
# STEP 4: Impute + scale (recommended for clustering)
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print('Clustering matrix shape:', X_scaled.shape)

In [ ]:
# STEP 5: Choose k using Elbow (inertia) + Silhouette score
k_values = range(2, 11)
inertias = []
sil_scores = []

silhouette_sample_size = min(10000, X_scaled.shape[0])

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(
        silhouette_score(
            X_scaled,
            labels,
            sample_size=silhouette_sample_size,
            random_state=42,
        )
    )

best_k = int(list(k_values)[int(np.argmax(sil_scores))])
print('Best k by silhouette:', best_k)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_values), inertias, marker='o')
axes[0].set_title('Elbow method (inertia)')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(k_values), sil_scores, marker='o')
axes[1].set_title(f'Silhouette score (sample={silhouette_sample_size})')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# STEP 6: Fit final K-Means, visualize, and save labels
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
clusters_kmeans = kmeans.fit_predict(X_scaled)

out_k = data.copy()
out_k['kmeans_cluster'] = clusters_kmeans

print('K-Means cluster counts:')
print(out_k['kmeans_cluster'].value_counts().sort_index())

# PCA 2D plot for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_kmeans, s=5, cmap='tab10', alpha=0.6)
plt.title(f'K-Means clusters (k={best_k}) visualized with PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

out_path_k = 'combined_cleaned_kmeans.csv'
out_k.to_csv(out_path_k, index=False)
print('Saved:', out_path_k)

## Agglomerative clustering (sampled) + dendrogram diagram

Agglomerative clustering is expensive for large datasets, so we fit it on a random sample (default up to 10,000 rows), then assign cluster labels for the full dataset using a nearest-centroid approximation.

The dendrogram below is generated using a small subset (default up to 50 sampled points), saved as `agglomerative_dendrogram.png`.

In [ ]:
# STEP 7: Agglomerative Clustering (sampled)
sample_size = min(10000, X_scaled.shape[0])
rng = np.random.RandomState(42)
sample_idx = rng.choice(X_scaled.shape[0], size=sample_size, replace=False)

X_sample = X_scaled[sample_idx]

# Using the same `best_k` selected by silhouette for K-Means
agg = AgglomerativeClustering(n_clusters=best_k, linkage='average')
labels_sample = agg.fit_predict(X_sample)

# Agglomerative dendrogram (hierarchical visualization; uses a small subset)
dendro_sample_size = min(50, X_sample.shape[0])
dendro_idx_local = rng.choice(X_sample.shape[0], size=dendro_sample_size, replace=False)
dendro_orig_idx = sample_idx[dendro_idx_local]

X_dendro = X_sample[dendro_idx_local]
Z = linkage(X_dendro, method='average', metric='euclidean')

plt.figure(figsize=(16, 6))
dendrogram(
    Z,
    labels=[str(i) for i in dendro_orig_idx],
    leaf_rotation=90,
    leaf_font_size=6,
)
plt.title('Agglomerative Cluster Dendrogram (sample)')
plt.xlabel('Sample index (from filtered dataset)')
plt.ylabel('Height')
plt.tight_layout()

out_dend_path = 'agglomerative_dendrogram.png'
plt.savefig(out_dend_path, dpi=200, bbox_inches='tight')
print('Saved:', out_dend_path)
plt.close()

In [ ]:
# Assign clusters to the full dataset (nearest centroid approximation)
centroids = np.vstack([
    X_sample[labels_sample == i].mean(axis=0)
    for i in range(best_k)
])

clusters_full, _ = pairwise_distances_argmin_min(
    X_scaled,
    centroids,
    metric='euclidean',
)

out2 = data.copy()
out2['agglomerative_cluster'] = clusters_full

print('Agglomerative cluster counts:')
print(out2['agglomerative_cluster'].value_counts().sort_index())

out_path2 = 'combined_cleaned_agglomerative.csv'
out2.to_csv(out_path2, index=False)
print('Saved:', out_path2)

### Cluster Visualization (BMI, glucose level, age)

Below are three scatter plots using the agglomerative labels:
- `BMI` vs `glucose`
- `BMI` vs `age`
- `glucose` vs `age`

In [ ]:
required_cols = ['bmi', 'glucose', 'age']
if all(c in out2.columns for c in required_cols):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].scatter(
        out2['bmi'],
        out2['glucose'],
        c=out2['agglomerative_cluster'],
        s=5,
        cmap='tab10',
        alpha=0.6,
    )
    axes[0].set_title('BMI vs Glucose')
    axes[0].set_xlabel('BMI')
    axes[0].set_ylabel('Glucose')

    axes[1].scatter(
        out2['bmi'],
        out2['age'],
        c=out2['agglomerative_cluster'],
        s=5,
        cmap='tab10',
        alpha=0.6,
    )
    axes[1].set_title('BMI vs Age')
    axes[1].set_xlabel('BMI')
    axes[1].set_ylabel('Age')

    axes[2].scatter(
        out2['glucose'],
        out2['age'],
        c=out2['agglomerative_cluster'],
        s=5,
        cmap='tab10',
        alpha=0.6,
    )
    axes[2].set_title('Glucose vs Age')
    axes[2].set_xlabel('Glucose')
    axes[2].set_ylabel('Age')

    plt.tight_layout()
    plt.show()
else:
    print('BMI/glucose/age columns not found; skipping scatter plots.')

In [ ]:
# Save a single merged CSV with both label sets
merged = data.copy()
merged['kmeans_cluster'] = out_k['kmeans_cluster'].values
merged['agglomerative_cluster'] = out2['agglomerative_cluster'].values

out_all = 'combined_cleaned_clusters_all.csv'
merged.to_csv(out_all, index=False)
print('Saved:', out_all)